# Benchmark Parallel Configurations

This notebook helps you find the optimal parallel processing configuration for your specific dataset and hardware.

## Features:
- Test different worker/batch combinations
- Measure performance and speed improvements
- Find optimal configuration automatically
- Compare CPU vs GPU performance

---

## 1. Setup and Dataset Analysis

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time

# Add scripts to path
import sys
sys.path.append('scripts')

# Import parallel evaluation utilities
from evaluation.parallel_evaluate_models import benchmark_parallel_configurations, get_optimal_config
from utils.timing_utils import initialize_timing, log_cell

# Initialize timing
notebook_start_time = time.time()
timer = initialize_timing("results/execution_logs/benchmark_timing.json")

# Analyze dataset
dataset_path = "data/test.csv"
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    dataset_size = len(df)
    print(f"📊 Dataset Analysis:")
    print(f"  Dataset: {dataset_path}")
    print(f"  Samples: {dataset_size}")
    print(f"  Columns: {list(df.columns)}")
    
    # Get optimal configs
    cpu_config = get_optimal_config(dataset_size, "cpu")
    gpu_config = get_optimal_config(dataset_size, "cuda")
    
    print(f"\n🔧 Recommended Configurations:")
    print(f"  CPU: {cpu_config['max_workers']} workers, {cpu_config['batch_size']} batch size")
    print(f"  GPU: {gpu_config['max_workers']} workers, {gpu_config['batch_size']} batch size")
else:
    print(f"❌ Dataset not found: {dataset_path}")
    dataset_size = 0

## 2. Benchmark Configurations

In [ ]:
def run_comprehensive_benchmark():
    """Run comprehensive benchmark of parallel configurations"""
    with log_cell("Comprehensive Parallel Benchmark"):
        if not os.path.exists(dataset_path):
            print(f"❌ Dataset not found: {dataset_path}")
            return
        
        # Define test configurations
        test_configs = [
            # Sequential baseline
            {"max_workers": 1, "batch_size": 1, "name": "Sequential"},
            
            # Small parallel
            {"max_workers": 2, "batch_size": 5, "name": "Small Parallel"},
            
            # Medium parallel
            {"max_workers": 3, "batch_size": 10, "name": "Medium Parallel"},
            
            # Large parallel
            {"max_workers": 4, "batch_size": 15, "name": "Large Parallel"},
            
            # Aggressive parallel
            {"max_workers": 6, "batch_size": 20, "name": "Aggressive Parallel"},
        ]
        
        # Test each model
        models_to_test = ["whisper", "wav2vec2"]  # Skip vosk for benchmarking
        all_results = {}
        
        for model_name in models_to_test:
            print(f"\n🔬 Benchmarking {model_name}...")
            
            # Convert configs for benchmark function
            benchmark_configs = [
                {"max_workers": c["max_workers"], "batch_size": c["batch_size"]}
                for c in test_configs
            ]
            
            try:
                results = benchmark_parallel_configurations(
                    model_name=model_name,
                    input_csv=dataset_path,
                    device="cpu",  # Use CPU for consistent benchmarking
                    vosk_model_path=None,
                    configurations=benchmark_configs
                )
                
                # Add config names
                for i, result in enumerate(results):
                    result["config_name"] = test_configs[i]["name"]
                
                all_results[model_name] = results
                
            except Exception as e:
                print(f"❌ Error benchmarking {model_name}: {e}")
                all_results[model_name] = []
        
        return all_results

# Run benchmark
benchmark_results = run_comprehensive_benchmark()

## 3. Analyze Benchmark Results

In [ ]:
def analyze_benchmark_results(results):
    """Analyze and visualize benchmark results"""
    with log_cell("Benchmark Results Analysis"):
        if not results:
            print("❌ No benchmark results to analyze")
            return
        
        # Create visualization
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Parallel Configuration Benchmark Results', fontsize=16, fontweight='bold')
        
        # 1. Speed comparison by model
        ax1 = axes[0, 0]
        for model_name, model_results in results.items():
            if model_results:
                configs = [r['config_name'] for r in model_results]
                speeds = [r['samples_per_second'] for r in model_results]
                ax1.plot(configs, speeds, marker='o', label=model_name, linewidth=2)
        
        ax1.set_title('Processing Speed by Configuration')
        ax1.set_ylabel('Samples per Second')
        ax1.legend()
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # 2. Execution time comparison
        ax2 = axes[0, 1]
        for model_name, model_results in results.items():
            if model_results:
                configs = [r['config_name'] for r in model_results]
                times = [r['total_time'] for r in model_results]
                ax2.plot(configs, times, marker='s', label=model_name, linewidth=2)
        
        ax2.set_title('Execution Time by Configuration')
        ax2.set_ylabel('Total Time (seconds)')
        ax2.legend()
        ax2.tick_params(axis='x', rotation=45)
        ax2.grid(True, alpha=0.3)
        
        # 3. Speedup vs Sequential
        ax3 = axes[1, 0]
        for model_name, model_results in results.items():
            if model_results:
                sequential_time = next((r['total_time'] for r in model_results if r['max_workers'] == 1), 1)
                configs = []
                speedups = []
                
                for r in model_results:
                    if r['max_workers'] > 1:
                        speedup = sequential_time / r['total_time']
                        configs.append(r['config_name'])
                        speedups.append(speedup)
                
                ax3.bar(configs, speedups, alpha=0.7, label=model_name)
        
        ax3.set_title('Speedup vs Sequential')
        ax3.set_ylabel('Speedup Factor')
        ax3.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='No Speedup')
        ax3.legend()
        ax3.tick_params(axis='x', rotation=45)
        ax3.grid(True, alpha=0.3)
        
        # 4. Efficiency (samples/second vs workers)
        ax4 = axes[1, 1]
        for model_name, model_results in results.items():
            if model_results:
                workers = [r['max_workers'] for r in model_results]
                speeds = [r['samples_per_second'] for r in model_results]
                ax4.scatter(workers, speeds, s=100, alpha=0.7, label=model_name)
                
                # Add labels for each point
                for i, (w, s) in enumerate(zip(workers, speeds)):
                    ax4.annotate(f"{model_results[i]['config_name']}", 
                               (w, s), 
                               xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        ax4.set_title('Efficiency: Workers vs Speed')
        ax4.set_xlabel('Number of Workers')
        ax4.set_ylabel('Samples per Second')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Save plots
        plot_path = f"results/benchmark_plots_{time.strftime('%Y%m%d_%H%M%S')}.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f"📊 Benchmark plots saved to: {plot_path}")
        
        return plot_path

# Analyze results
if benchmark_results:
    plot_path = analyze_benchmark_results(benchmark_results)
else:
    print("❌ No benchmark results available")

## 4. Generate Recommendations

In [ ]:
def generate_recommendations(results, dataset_size):
    """Generate configuration recommendations based on benchmark results"""
    with log_cell("Generate Configuration Recommendations"):
        if not results:
            print("❌ No results to analyze for recommendations")
            return
        
        recommendations = {}
        
        for model_name, model_results in results.items():
            if not model_results:
                continue
            
            # Find best configurations
            fastest = min(model_results, key=lambda x: x['total_time'])
            most_efficient = max(model_results, key=lambda x: x['samples_per_second'])
            best_speedup = max([r for r in model_results if r['max_workers'] > 1], 
                            key=lambda x: x['total_time'])
            
            sequential_time = next((r['total_time'] for r in model_results if r['max_workers'] == 1), None)
            max_speedup = sequential_time / fastest['total_time'] if sequential_time else 1.0
            
            recommendations[model_name] = {
                'fastest_config': fastest,
                'most_efficient_config': most_efficient,
                'max_speedup': max_speedup,
                'sequential_time': sequential_time,
                'fastest_time': fastest['total_time']
                'time_saved': sequential_time - fastest['total_time'] if sequential_time else 0
            }
        
        # Print recommendations
        print("\n🎯 Configuration Recommendations:")
        print("="*60)
        
        for model_name, rec in recommendations.items():
            print(f"\n{model_name.upper()}:")
            print(f"  🏆 Fastest: {rec['fastest_config']['config_name']} ({rec['fastest_config']['max_workers']} workers, {rec['fastest_config']['batch_size']} batch)")
            print(f"     Time: {rec['fastest_time']:.1f}s (vs {rec['sequential_time']:.1f}s sequential)")
            print(f"     Time saved: {rec['time_saved']:.1f}s ({rec['max_speedup']:.1f}x speedup)")
            print(f"  ⚡ Most Efficient: {rec['most_efficient_config']['config_name']} ({rec['most_efficient_config']['samples_per_second']:.2f} samples/s)")
        
        # Overall recommendations
        print(f"\n📋 Overall Recommendations for {dataset_size} samples:")
        
        # Find best overall configuration
        all_configs = []
        for model_name, rec in recommendations.items():
            config = rec['fastest_config'].copy()
            config['model'] = model_name
            config['speedup'] = rec['max_speedup']
            all_configs.append(config)
        
        if all_configs:
            best_overall = min(all_configs, key=lambda x: x['total_time'])
            print(f"  🥇 Best overall: {best_overall['model']} with {best_overall['config_name']}")
            print(f"     Config: {best_overall['max_workers']} workers, {best_overall['batch_size']} batch size")
            print(f"     Expected time: {best_overall['total_time']:.1f}s")
        
        # Hardware-specific recommendations
        print(f"\n💻 Hardware-Specific Tips:")
        if dataset_size < 50:
            print(f"  Small dataset (<50 samples): 2-3 workers, batch size 5-10")
        elif dataset_size < 200:
            print(f"  Medium dataset (50-200 samples): 3-4 workers, batch size 10-15")
        else:
            print(f"  Large dataset (>200 samples): 4-6 workers, batch size 15-25")
        
        print(f"\n🔧 Recommended CONFIG settings:")
        for model_name, rec in recommendations.items():
            fastest = rec['fastest_config']
            print(f"  {model_name}: max_workers={fastest['max_workers']}, batch_size={fastest['batch_size']}")
        
        return recommendations

# Generate recommendations
if benchmark_results:
    recommendations = generate_recommendations(benchmark_results, dataset_size)
else:
    print("❌ No benchmark results available for recommendations")

## 5. Export Benchmark Report

In [ ]:
def export_benchmark_report(results, recommendations, plot_path):
    """Export comprehensive benchmark report"""
    with log_cell("Export Benchmark Report"):
        if not results:
            print("❌ No benchmark results to export")
            return
        
        timestamp = time.strftime('%Y%m%d_%H%M%S')
        
        # Create comprehensive report
        report = {
            "benchmark_timestamp": time.strftime('%Y-%m-%d %H:%M:%S'),
            "dataset_size": dataset_size,
            "dataset_path": dataset_path,
            "all_results": results,
            "recommendations": recommendations,
            "summary": {
                "models_tested": list(results.keys()),
                "configurations_tested": len(next(iter(results.values()))) if results else 0,
                "best_speedup": max([rec['max_speedup'] for rec in recommendations.values()]) if recommendations else 0
            }
        }
        
        # Save JSON report
        json_path = f"results/benchmark_report_{timestamp}.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        
        # Save CSV summary
        all_data = []
        for model_name, model_results in results.items():
            for result in model_results:
                result['model'] = model_name
                all_data.append(result)
        
        if all_data:
            df = pd.DataFrame(all_data)
            csv_path = f"results/benchmark_summary_{timestamp}.csv"
            df.to_csv(csv_path, index=False)
            
            print(f"\n📋 Benchmark Report Generated:")
            print(f"  📄 JSON Report: {json_path}")
            print(f"  📊 CSV Summary: {csv_path}")
            print(f"  📈 Plots: {plot_path}")
            
            return json_path, csv_path, plot_path
        else:
            print("❌ No data to export")
            return None, None, None

# Export report
if benchmark_results and 'recommendations' in locals():
    report_files = export_benchmark_report(benchmark_results, recommendations, plot_path)
else:
    print("❌ No benchmark results available for export")

## 6. Final Summary

### How to Use Benchmark Results:

1. **Update Your CONFIG**: Use the recommended settings in your main evaluation notebook
   ```python
   CONFIG = {
       "parallel_enabled": True,
       "max_workers": 3,  # From benchmark recommendation
       "batch_size": 10,   # From benchmark recommendation
       # ... other settings
   }
   ```

2. **Expected Performance Gains**:
   - **Small datasets** (<50 samples): 2-3x speedup
   - **Medium datasets** (50-200 samples): 3-4x speedup
   - **Large datasets** (>200 samples): 4-6x speedup

3. **Hardware Considerations**:
   - **CPU**: More workers generally better
   - **GPU**: Fewer workers due to memory constraints
   - **Memory**: Monitor usage with large batch sizes

4. **Model-Specific Tips**:
   - **Whisper**: Benefits from moderate parallelization
   - **wav2vec2**: Good speedup with parallel processing
   - **Vosk**: Lightweight, can use more workers

### Quick Configuration Guide:

| Dataset Size | Workers | Batch Size | Expected Speedup |
|-------------|---------|------------|-----------------|
| < 50       | 2-3     | 5-10       | 2-3x           |
| 50-200     | 3-4     | 10-15      | 3-4x           |
| > 200      | 4-6     | 15-25      | 4-6x           |

---

**🎯 Ready to optimize your ASR evaluation with parallel processing!**